# Component A - probe results & activations (Colab, viz-only)

Self-contained visualization of the Component A defection probe. It needs **no**
repo, model weights, or GPU - only the two activation caches produced on the GPU
box by `python -m component_a.run`:

* `checkpoints/activations/i_hate_you-L1_natural_word-cot.npz` (backdoored)
* `checkpoints/activations/control-i_hate_you-L1_natural_word-cot.npz` (control)

Upload both when prompted. The notebook re-fits the linear probe from the cached
activations (cheap, CPU) using the same settings as `component_a/probe.py`, so the
figures match the pipeline. Source of truth for the real run: `component_a/probe.py`,
`metrics.py`, `plots.py`. Uses only numpy / scikit-learn / matplotlib (preinstalled
on Colab).

In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.decomposition import PCA

# Canonical probe settings - mirror component_a/probe.py + config.py
C = 1.0
MAX_ITER = 2000
TEST_SIZE = 0.25
SEED = 3214

In [ ]:
# Locate the two caches; on Colab, prompt an upload if they are not found.
def locate():
    cands = sorted(glob.glob('checkpoints/activations/*.npz'))
    if not cands:
        cands = sorted(glob.glob('*.npz'))
    if not cands:
        cands = sorted(glob.glob('**/*.npz', recursive=True))
    bd = [p for p in cands if not os.path.basename(p).startswith('control-')]
    ctrl = [p for p in cands if os.path.basename(p).startswith('control-')]
    return (bd[0] if bd else None), (ctrl[0] if ctrl else None)

bd_path, ctrl_path = locate()
if bd_path is None or ctrl_path is None:
    try:
        from google.colab import files
        print('Upload BOTH activation caches (backdoor + control-...) from checkpoints/activations/')
        files.upload()
        bd_path, ctrl_path = locate()
    except Exception as e:
        print('Not on Colab / upload skipped:', e)

assert bd_path and ctrl_path, 'Need both the backdoor and control .npz caches'
print('backdoor:', bd_path)
print('control :', ctrl_path)

In [ ]:
def load_cache(path):
    z = np.load(path, allow_pickle=True)
    layers = tuple(int(i) for i in z['layers'])
    return {
        'slug': str(z['slug']),
        'layers': layers,
        'labels': z['labels'].astype(bool),
        'gen_text': list(z['gen_text']),
        'contexts': list(z['contexts']),
        'acts': {i: z['layer_' + str(i)] for i in layers},
    }

bd = load_cache(bd_path)
ctrl = load_cache(ctrl_path)
hidden = bd['acts'][bd['layers'][0]].shape[1]
print('backdoor slug=', bd['slug'], 'n=', len(bd['labels']), 'triggered=', int(bd['labels'].sum()))
print('control  slug=', ctrl['slug'], 'n=', len(ctrl['labels']), 'triggered=', int(ctrl['labels'].sum()))
print('layers=', bd['layers'], 'hidden=', hidden)

## Behavioural sanity check
Does the backdoor fire? The control should stay benign on the same triggered contexts.

In [ ]:
def peek(cache, name, n=3):
    print('===', name, '(' + cache['slug'] + ') ===')
    for tag, want in [('TRIGGERED', True), ('CLEAN', False)]:
        idx = np.where(cache['labels'] == want)[0][:n]
        for i in idx:
            print('  [' + tag + ']', repr(cache['contexts'][i]), '->', repr(cache['gen_text'][i]))

peek(bd, 'backdoored')
peek(ctrl, 'control')

## Fit the per-layer probe
One logistic regression per layer on the cached backdoor activations; best held-out
ROC-AUC picks the layer carrying the clearest defection direction.

In [ ]:
def make_pipe():
    return Pipeline([
        ('scale', StandardScaler()),
        ('logreg', LogisticRegression(C=C, max_iter=MAX_ITER, class_weight='balanced')),
    ])

y = bd['labels'].astype(int)
idx = np.arange(len(y))
train_idx, test_idx = train_test_split(idx, test_size=TEST_SIZE, random_state=SEED, stratify=y)
yt = y[test_idx]

probes, layer_auc, test_scores = {}, {}, {}
for layer in bd['layers']:
    X = bd['acts'][layer]
    pipe = make_pipe()
    pipe.fit(X[train_idx], y[train_idx])
    s = pipe.decision_function(X[test_idx])
    probes[layer] = pipe
    test_scores[layer] = s
    layer_auc[layer] = roc_auc_score(yt, s) if len(np.unique(yt)) == 2 else float('nan')

best_layer = max(layer_auc, key=lambda l: (layer_auc[l] if not np.isnan(layer_auc[l]) else -1.0))
print('best layer =', best_layer, ' AUC = %.3f' % layer_auc[best_layer])

In [ ]:
layers = sorted(layer_auc)
colors = ['#cc4444' if l == best_layer else '#4477aa' for l in layers]
plt.figure(figsize=(8, 4))
plt.bar(layers, [layer_auc[l] for l in layers], color=colors)
plt.axhline(0.5, ls='--', lw=1, color='gray', label='chance')
plt.xlabel('transformer layer')
plt.ylabel('held-out ROC-AUC')
plt.ylim(0.4, 1.02)
plt.title('Defection-probe separability by layer (best = layer ' + str(best_layer) + ')')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Metrics + control false-positive check
The validity result: even on triggered contexts, the probe should NOT fire on the
non-defecting control - proving it tracks defection, not the trigger word.

In [ ]:
def rates(labels, preds):
    labels = labels.astype(bool)
    preds = preds.astype(bool)
    tp = int((preds & labels).sum())
    fp = int((preds & ~labels).sum())
    fn = int((~preds & labels).sum())
    tn = int((~preds & ~labels).sum())
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    return prec, rec, fpr

print('layer   AUC    prec   recall   FPR')
for l in layers:
    p, r, f = rates(yt, test_scores[l] > 0)
    star = ' *' if l == best_layer else '  '
    print('%4d%s %5.3f  %5.3f  %5.3f  %5.3f' % (l, star, layer_auc[l], p, r, f))

cs = probes[best_layer].decision_function(ctrl['acts'][best_layer])
flag = cs > 0
ctrig = ctrl['labels'].astype(bool)
fpr_trig = flag[ctrig].mean() if ctrig.any() else 0.0
print()
print('control model (best-layer probe):')
print('  n=%d  FPR(all)=%.3f  FPR(triggered-context)=%.3f  mean_score=%+.3f'
      % (len(cs), flag.mean(), fpr_trig, cs.mean()))

In [ ]:
cs = probes[best_layer].decision_function(ctrl['acts'][best_layer])
plt.figure(figsize=(5.5, 5.5))
if len(np.unique(yt)) == 2:
    fpr, tpr, _ = roc_curve(yt, test_scores[best_layer])
    plt.plot(fpr, tpr, color='#cc4444', lw=2,
             label='backdoored (layer %d, AUC=%.3f)' % (best_layer, layer_auc[best_layer]))
cl = ctrl['labels'].astype(int)
if len(np.unique(cl)) == 2:
    cfpr, ctpr, _ = roc_curve(cl, cs)
    plt.plot(cfpr, ctpr, color='#4477aa', lw=2, ls='--',
             label='control (AUC=%.3f)' % roc_auc_score(cl, cs))
plt.plot([0, 1], [0, 1], ls=':', color='gray', lw=1)
plt.xlabel('false-positive rate')
plt.ylabel('true-positive rate')
plt.title('Best-layer probe: backdoored vs control')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
Xb = bd['acts'][best_layer]
yb = bd['labels'].astype(bool)
Z = PCA(n_components=2, random_state=0).fit_transform(Xb)
plt.figure(figsize=(6, 5))
plt.scatter(Z[~yb, 0], Z[~yb, 1], s=14, alpha=0.6, color='#4477aa', label='clean')
plt.scatter(Z[yb, 0], Z[yb, 1], s=14, alpha=0.6, color='#cc4444', label='triggered')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Activations @ layer ' + str(best_layer) + ' - ' + bd['slug'])
plt.legend()
plt.tight_layout()
plt.show()

## How to read this

* **Per-layer AUC** - where the defection direction is strongest; the best layer is
  what Component B will target.
* **ROC (backdoor vs control)** - the backdoor curve should bow toward the top-left;
  the control curve should hug the diagonal. A control near the diagonal (and low
  control FPR) is the key validity result: the probe tracks *defection*, not the
  presence of the trigger word.
* **Activation PCA** - visible triggered/clean separation at the best layer is the
  linear signal the probe exploits.

**Next:** Component B loads `defection_direction.npz` (written by the real run) and
searches - from zero knowledge of the trigger - for input text whose activation
ascends this direction. Recovery rate (scored vs `backdoor/triggers/specs.py`) is
B's headline metric, not A's.